## tl;dr

方向预测适合做**报价偏移和逆向选择过滤**，不适合直接变成吃单信号。本实验使用严格 MBO 重建，在 07/09 验证、07/21 测试，再以 07/21 验证、08/07 测试。分类准确率分别为 58.85% 和 58.48%。

风险控制开关只在先前验证期胜过无信号基线时启用 alpha：第一阶段启用，第二阶段关闭。在乐观的 touch 排队情景、每边 2.77 bps 成本下，两天方向增强策略合计 +181.90 HKD，传统基线 -1,758.26 HKD；但只有 1/2 天盈利。在保守的 through 情景下，方向增强策略仍为 -1,737.17 HKD（基线 -3,827.12 HKD）。因此它是值得继续验证的做市框架，但尚不是可上线的赚钱策略。

## Context & Methods

模型把 `p(up)-p(down)` 在先前验证期校准为预期未来中价 tick，形成方向项。离散保留价为：

`reservation center = mid + signal_strength × expected_move - inventory_penalty`

买卖报价围绕该中心移动，但保持被动；库存达到硬上限后停止继续增加同向库存，测试结束按一档价主动平仓。参数只在更早的验证期、使用保守 through 成交假设选择。

### Key Assumptions

- 一个 quote interval 为 20 个采样快照；假设 5 ms 下单延迟。
- 每次成交 100 股；官方固定费用 1.27 bps/边，另假设佣金 1.50 bps/边。
- touch：成交价到达挂单价即视为成交（乐观）；through：至少穿过一档才视为成交（保守）。
- 数据没有虚拟订单的真实队列位置，两个情景只能作为上下敏感性分析。
- 完整复现命令：`python 0824/run_directional_market_maker.py`。

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'output').exists():
    ROOT = ROOT / '0824'
RESULT_PATH = ROOT / 'output' / 'directional_market_maker_results.json'
VALIDATION_PATH = ROOT / 'output' / 'directional_market_maker_validation.json'
results = json.loads(RESULT_PATH.read_text(encoding='utf-8'))
validation = json.loads(VALIDATION_PATH.read_text(encoding='utf-8'))
results['as_of'], validation['all_checks_passed']

## Data

只使用三天未污染的严格 MBO 订单级重建；成交判定来自原始 MsgType=50 成交打印。

In [ ]:
quality = pd.DataFrame(results['data_quality']).T
quality.index.name = 'date'
quality[['book_snapshots', 'trade_prints', 'mbo_state_tainted', 'order_errors']]

## Results

In [ ]:
rows = []
for experiment in results['experiments']:
    for fill_mode in ('through', 'touch'):
        for strategy in ('traditional_baseline', 'directional', 'paired_no_signal_ablation'):
            record = experiment['test']['strategies'][strategy][fill_mode]
            rows.append({
                'test_date': experiment['test_date'],
                'accuracy': experiment['test_classification']['accuracy'],
                'signal_enabled': experiment['signal_activation']['enabled'],
                'fill_mode': fill_mode,
                'strategy': strategy,
                'maker_fills': record['maker_fills'],
                'gross_pnl_hkd': record['gross_pnl_hkd'],
                'official_only_net_hkd': record['net_pnl_official_only_hkd'],
                'all_in_net_hkd': record['net_pnl_hkd'],
            })
detail = pd.DataFrame(rows)
detail.round(3)

In [ ]:
aggregate_rows = []
for fill_mode, strategies in results['aggregate_test_results'].items():
    for strategy, record in strategies.items():
        aggregate_rows.append({
            'fill_mode': fill_mode,
            'strategy': strategy,
            'gross_hkd': record['total_gross_pnl_hkd'],
            'official_only_net_hkd': record['total_net_pnl_official_only_hkd'],
            'all_in_net_hkd': record['total_net_pnl_hkd'],
            'profitable_days': record['profitable_days'],
        })
aggregate = pd.DataFrame(aggregate_rows)
aggregate.round(2)

In [ ]:
pd.Series(validation['checks'], name='passed').to_frame(), validation['assessment']

## Takeaways

1. 方向信息确实能用于做市：在启用信号的 07/21 测试中，它在 touch 和 through 两种成交假设下都显著优于两个无信号对照。
2. 60% 左右分类准确率不是稳定 alpha 的充分条件。07/21 的经济校准较弱，风险开关因此在 08/07 关闭信号。
3. 费用决定了结论：touch 情景含假设佣金后两天合计仅 +181.90 HKD；through 情景即使方向增强仍亏损。
4. 当前结论为 `Share with caveats`：可以作为 mentor 讨论和下一阶段实验的模型原型，但需真实队列位置、更多连续交易日和实际被动费率后才能判断是否可赚钱。